# Model Tester

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np
import h5py

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# os.environ["NCCL_DEBUG"] = "INFO"
# os.environ["NCCL_DEBUG_SUBSYS"] = "ALL"

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Flatten,
    LeakyReLU,
    Permute,
    LayerNormalization,
    Activation,
    Concatenate,
    Conv1D,
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.metrics import RootMeanSquaredError

from mlpng import Core
from mlpng.utils import (
    setup_logging,
    get_fisher,
    plot_predictions,
    plot_histogram,
    print_errors,
    plot_metrics,
    try_init_wandb,
)
from mlpng.utils.dataloaders import tfds_from_hdf5, tfds_from_hdf5_lens

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import (
    HealpyChebyshev,
    HealpyPool,
    HealpyPseudoConv_Transpose,
)


os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
keras = tf.keras

In [3]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print("Python executable:", sys.executable)
print(f"TensorFlow version: {tf.__version__}")
# print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
# print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")
# print(f"keras version: {keras.__version__}")

Conda environment: ds25
Python executable: /users/stevensonb/.conda/envs/ds25/bin/python
TensorFlow version: 2.16.2


In [4]:
logger = setup_logging(__name__, level=logging.DEBUG)

In [5]:
class SCNBlock(keras.Model):  # type: ignore

    def __init__(
        self,
        filters,
        n_mid=None,
        n_mid_scale=4,
        n_out=None,
        n_neighbors=8,
        cheb_degree=2,
        cheb_init=None,
        cheb_act=None,
        cheb_bias=True,
        cheb_batch=True,
        batch_size=None,
        activation="relu",
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.filters = filters
        self.n_neighbors = n_neighbors
        if n_mid is None:
            n_mid = filters * n_mid_scale
        if n_out is None:
            n_out = filters
        self.n_mid = n_mid
        self.n_out = n_out
        self.cheb_degree = cheb_degree
        self.cheb_init = cheb_init
        self.cheb_act = cheb_act
        self.cheb_bias = cheb_bias
        self.cheb_batch = cheb_batch
        self.batch_size = batch_size
        self.activation = activation

        self.res_conv_1d = Conv1D(self.filters, kernel_size=1)
        self.cheb = HealpyChebyshev(
            K=self.cheb_degree,
            initializer=self.cheb_init,
            activation=self.cheb_act,
            use_bias=self.cheb_bias,
            use_bn=self.cheb_batch,
            depth_wise=True,
        )

    def build(self, input_shape):
        # super().build(input_shape)

        npix = input_shape[-2]
        nside = hp.npix2nside(npix)
        indices = np.arange(npix)

        self.res_conv_1d.build(input_shape)

        self.gcnn = HealpyGCNN(
            int(nside),
            indices,
            layers=[
                self.cheb,
                Conv1D(self.n_mid, kernel_size=1),
                LayerNormalization(),
                Activation(self.activation),
                Conv1D(self.n_out, kernel_size=1),
            ],
            verbose=False,
            max_batch_size=self.batch_size,
            initial_Fin=input_shape[-1],
        )
        self.gcnn.build(input_shape)

    def call(self, inputs, training=None, mask=None):
        res = inputs
        if res.shape[-1] != self.filters:
            res = self.res_conv_1d(res)

        x = self.gcnn(inputs)
        return x + res

    def compute_output_shape(self, input_shape):
        output = (input_shape[0], input_shape[1], self.n_out)
        return output


# @tf.keras.saving.register_keras_serializable()
def SCNUNet(
    input_shape,
    base_channels=2,
    n_neighbors=8,
    cheb_degree=3,
    cheb_init=None,
    cheb_act=None,
    cheb_bias=True,
    cheb_batch=True,
    max_batch_size=64,
    n_bottleneck=1,
    token_dim=16,
    channel_dim=16,
    name="SCNUNet",
    scn_act="relu",
):
    nside = hp.npix2nside(input_shape[-2])
    n_layers = math.floor(math.log(nside, 2))
    channels = [base_channels * 2**p for p in range(n_layers)]

    inputs = keras.Input(shape=input_shape)
    x = inputs

    skips = []
    for layer in range(n_layers):
        x = SCNBlock(
            channels[layer],
            n_neighbors=n_neighbors,
            cheb_degree=cheb_degree,
            cheb_init=cheb_init,
            cheb_act=cheb_act,
            cheb_bias=cheb_bias,
            cheb_batch=cheb_batch,
            batch_size=max_batch_size,
            activation=scn_act,
        )(x)
        skips.append(x)
        x = HealpyPool(p=1, pool_type="AVG")(x)

    # bottleneck
    for i in range(n_bottleneck):
        x = mixer_block(token_dim, channel_dim, name=f"Bottleneck_{i}")(x)
    if n_bottleneck > 0:
        x = layers.LayerNormalization(name="Bottleneck_norm")(x)

    for block, skip in zip(reversed(range(n_layers)), reversed(skips)):
        x = HealpyPseudoConv_Transpose(1, channels[block])(x)
        x = Concatenate()([x, skip])

        x = SCNBlock(
            channels[block],
            n_neighbors=n_neighbors,
            cheb_degree=cheb_degree,
            cheb_init=cheb_init,
            cheb_act=cheb_act,
            cheb_bias=cheb_bias,
            cheb_batch=cheb_batch,
            batch_size=max_batch_size,
        )(x)

    x = Conv1D(input_shape[-1], 1)(x)
    return keras.Model(inputs=inputs, outputs=x, name=name)  # type: ignore


# @tf.keras.saving.register_keras_serializable()  # type: ignore
class mlp_block(layers.Layer):
    def __init__(self, hidden_dim=512, activation="gelu", name="MLPBlock"):
        super().__init__(name=name)
        self.hidden_dim = hidden_dim
        self.activation = activation

    def build(self, input_shape):
        self.d1 = layers.Dense(self.hidden_dim, activation=self.activation)
        self.d2 = layers.Dense(input_shape[-1])

    def call(self, inputs):
        x = self.d1(inputs)
        return self.d2(x)

    def get_config(self):
        return {
            "hidden_dim": self.hidden_dim,
            "activation": self.activation,
            "name": self.name,
        }


# @tf.keras.saving.register_keras_serializable()  # type: ignore
class mixer_block(layers.Layer):
    def __init__(
        self,
        token_dim=256,
        channel_dim=2048,
        name="MLPMixerBlock",
        activation="gelu",
    ):
        super().__init__(name=name)
        self.token_dim = token_dim
        self.channel_dim = channel_dim
        self.activation = activation

        self.norm = layers.LayerNormalization()
        self.perm_1 = layers.Permute((2, 1))
        self.perm_2 = layers.Permute((2, 1))
        self.token_mixing = mlp_block(
            token_dim, activation=activation, name="TokenMixing"
        )
        self.channel_mixing = mlp_block(
            channel_dim, activation=activation, name="ChannelMixing"
        )

    def call(self, inputs):
        y = self.norm(inputs)
        y = self.perm_1(y)
        y = self.perm_2(self.token_mixing(self.perm_1(y)))
        y = self.perm_2(y)
        x = inputs + y
        y = self.norm(x)
        return x + self.channel_mixing(y)

    def get_config(self):
        return {
            "token_dim": self.token_dim,
            "channel_dim": self.channel_dim,
            "activation": self.activation,
            "name": self.name,
        }


def get_model(
    full_model,
    ff_layers=3,
    token_dim=64,
    channel_dim=512,
    activation="relu",
    name="SCNReg",
):
    encoder_input = full_model.input
    encoder_output = None
    for layer in full_model.layers:
        if "Bottleneck_0" == layer.name:
            encoder_output = layer.output
            break

    encoder_model = keras.Model(inputs=encoder_input, outputs=encoder_output)  # type: ignore
    encoder_model.trainable = False

    x = encoder_model.output
    for i in range(ff_layers):
        x = mixer_block(
            token_dim,
            channel_dim,
            activation=activation,
            name=f"MLPMixerBlock_{i}",
        )(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = Dense(1, kernel_initializer="zeros")(x)
    return keras.Model(inputs=encoder_input, outputs=x, name=name)  # type: ignore

In [8]:
core = Core(
    [
        "settings/n32.json",
        "--nsims",
        "10000",
        "--pols",
        "T",
    ]
)

core.init_estimator()

10-Feb-25 14:43:52 - mlpng.core - INFO - Parsing CLI args: ['settings/n32.json', '--nsims', '10000', '--pols', 'T']
10-Feb-25 14:43:52 - mlpng.core - INFO - Loading settings from file 'settings/n32.json'
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Forcing setting 'nsims' to 10000 due to CLI
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Forcing setting 'pols' to ['T'] due to CLI
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 67.4, 'As': 2.457e-09, 'ns': 0.9649, 'ombh2': 0.0224, 'omch2': 0.12, 'tau': 0.054} (default: {'As': 2.13e-09, 'ns': 0.9624, 'pivot_scalar': 0.05})
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.457e-09
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Overriding cosmo param 'ns' from 0.9624 to 0.9649
10-Feb-25 14:43:52 - mlpng.core - INFO - Running with settings: 
{
  "cosmo_params": {
    "As": 2.457e-09,
    "ns": 0.9649,
    "pivot_scalar": 0.05,
    "H0": 67.4,
    "ombh2": 0.0224,
    "om

10-Feb-25 14:43:52 - mlpng.core - DEBUG - Setting 'data_dir' not found, using default: 'data'
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Setting 'tb_dir' not found, using default: 'tensorboard'
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Setting 'model_dir' not found, using default: 'models'
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Setting 'mc_dir' not found, using default: 'kswmc'
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Using data file: data/data/l95_n32_T_10000x25_f100.hdf5
10-Feb-25 14:43:52 - mlpng.core - DEBUG - Setting 'mc_steps' not found, using default: 100


ModuleNotFoundError: No module named 'camb'

In [ ]:
BATCH_SIZE = 256
# train, val, test = tfds_from_hdf5(
#     core,
#     "/map",
#     "/map",
#     batch_size=BATCH_SIZE,
#     reshape_y=False,
# )

train, val, test = tfds_from_hdf5_lens(
    core,
    "/alm",
    "/alm",
    batch_size=BATCH_SIZE,
    reshape_y=False,
    transpose_y=True,
    buffer=1000,
    lens_scale=10.0,
)

AttributeError: in user code:

    File "/users/stevensonb/Research/MLPNG/mlpng/utils/dataloaders.py", line 92, in None  *
        lambda x, y: lens_ds(x, y, core, lens_scale, unet=y_name == "/alm")
    File "/users/stevensonb/Research/MLPNG/mlpng/utils/dataloaders.py", line 38, in lens_ds  *
        cl_phi = core.cosmo._camb_data.get_lens_potential_cls(core.lmax, "muK", True)

    AttributeError: 'Core' object has no attribute 'cosmo'


In [ ]:
fisher = get_fisher(core.file, "fisher_iso")
npix = hp.nside2npix(core.nside)
decay_steps = core.total_sims * 0.8 // BATCH_SIZE  # once per epoch

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    learning_rate = ExponentialDecay(1e-2, decay_steps, 0.95, staircase=True)

    unet = SCNUNet((npix, core.npols))
    unet.compile(
        optimizer=AdamW(learning_rate, weight_decay=0.1),  # type: ignore
        loss="mse",
        metrics=[RootMeanSquaredError()],  # type: ignore
    )

unet.summary()

In [ ]:
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
]

history = unet.fit(
    train,
    epochs=20,
    validation_data=val,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
truth = np.concatenate([y for _, y in test])
preds = unet.predict(test, verbose=0)

# using evaluate to get the rmse since manually calculating it was giving a different value
rmse = unet.evaluate(test, verbose=0)[1]
print_errors(truth, preds, fisher)

In [ ]:
plot_metrics(history, metrics=["loss"], show=True)
# plot_predictions(truth, preds, fisher=fisher, title=f"RMSE: {rmse:.3f}", show=True)
# plot_histogram(truth, preds, show=True)

## Second part

In [ ]:
BATCH_SIZE = 128
# train, val, test = tfds_from_hdf5(
#     core,
#     "/map",
#     "/fnl",
#     batch_size=BATCH_SIZE,
# )

train, val, test = tfds_from_hdf5_lens(
    core,
    "/alm",
    "/fnl",
    batch_size=BATCH_SIZE,
    lens_scale=10.0,
)

In [ ]:
decay_steps = core.total_sims * 0.8 // BATCH_SIZE  # once per epoch

with strategy.scope():
    learning_rate = ExponentialDecay(4e-3, decay_steps, 0.95, staircase=False)
    # learning_rate = LinearWarmup(learning_rate, decay_steps, 1e-8)

    model = get_model(unet)
    model.build((npix, 1))
    model.compile(
        optimizer=AdamW(learning_rate, global_clipnorm=1),  # type: ignore
        loss="mse",
        metrics=[RootMeanSquaredError()],  # type: ignore
    )

model.summary()

In [ ]:
tf_dir = f"{core.dirs['tb']}/{core.name}-reg"
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
]

history = model.fit(
    train,
    epochs=300,
    validation_data=val,
    callbacks=callbacks,
    verbose=1,
)

model.evaluate(test, verbose=2)  # type: ignore

preds = model.predict(test, verbose=2).flatten()  # * 100  # type: ignore
truth = np.concatenate([y for _, y in test])  # * 100  # type: ignore

In [ ]:
print_errors(truth, preds, fisher)
plot_metrics(history, metrics=["loss"], show=True)
plot_predictions(
    truth,
    preds,
    fisher=fisher,
    show=True,
)
plot_histogram(truth, preds, show=True)